# GRU-D H1 to joint M4 V2 — recovered continuation

This private notebook restores the hash-locked step-2500 checkpoint from cancelled Notebook v4 and continues the original global schedule to step 4000. The old checkpoint lacks RNG state, so the first continuation is reproducible but not bitwise identical to an uninterrupted run.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json

RESUME_SCHEMA = 'trauma_predict.grud_h1_joint_m4_resume_bundle.v1'
RESUME_REF = 'vanila111/trauma-predict-grud-h1-joint-m4-v2-resume-2500-bundle'
matches = []
for manifest_path in sorted(Path('/kaggle/input').rglob('grud_v2_resume_bundle_manifest.json')):
    if manifest_path.is_symlink() or not manifest_path.is_file():
        continue
    payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    if payload.get('schema') == RESUME_SCHEMA and payload.get('dataset_ref') == RESUME_REF:
        matches.append((manifest_path.parent.resolve(), payload))
if len(matches) != 1:
    raise RuntimeError(f'Expected one pre-bound resume Input, found {len(matches)}')
bundle, manifest = matches[0]
launcher_row = manifest.get('launcher')
if not isinstance(launcher_row, dict):
    raise TypeError('Resume manifest lacks launcher lock')
launcher = bundle / str(launcher_row.get('path') or '')
digest = hashlib.sha256(launcher.read_bytes()).hexdigest() if launcher.is_file() else ''
if launcher.is_symlink() or launcher.stat().st_size != int(launcher_row.get('size_bytes', -1)) or digest != launcher_row.get('sha256'):
    raise ValueError('Mounted resume launcher differs from its manifest')
spec = importlib.util.spec_from_file_location('grud_v2_resume_bootstrap', launcher)
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
RESUME_STATE = bootstrap.bind_resume_inputs()


In [ ]:
RUNTIME_ENVIRONMENT = bootstrap.install_p100_runtime()

In [ ]:
bootstrap.run_training(RESUME_STATE, RUNTIME_ENVIRONMENT)